In [ ]:
import pandas as pd
import os
from datetime import date

def scan_and_process_customers():
    path = '/content/'
    files = [f for f in os.listdir(path) if f.endswith('.csv')]

    # Schema mục tiêu
    target_cols = ['customer_id', 'zip_FK', 'signup_date', 'gender', 'age_group', 'acq_channel']

    # Mapping dự đoán các tên cột tương đồng
    potential_mappings = {
        'customer_id': ['customer_id', 'cust_id', 'user_id', 'id'],
        'zip_FK': ['zip', 'zip_code', 'postal_code', 'zip_FK'],
        'signup_date': ['signup_date', 'created_at', 'join_date', 'registration_date'],
        'gender': ['gender', 'sex'],
        'age_group': ['age_group', 'age_bracket', 'age'],
        'acq_channel': ['acq_channel', 'channel', 'acquisition', 'source', 'acquisition_channel']
    }

    source_report = {}
    collected_dfs = []

    print("--- Bắt đầu quét các file cho bảng CUSTOMER ---")
    for file in files:
        if file == 'customers_new.csv': continue
        try:
            file_path = os.path.join(path, file)
            # Sử dụng utf-8-sig để đọc đúng tiếng Việt nếu có
            temp_df = pd.read_csv(file_path, nrows=0, encoding='utf-8-sig')
            found_cols = {}

            for target, aliases in potential_mappings.items():
                for alias in aliases:
                    if alias in temp_df.columns:
                        found_cols[alias] = target
                        break

            if len(found_cols) >= 2:
                print(f"Tìm thấy dữ liệu CUSTOMER trong: {file} ({list(found_cols.keys())})")
                full_df = pd.read_csv(file_path, encoding='utf-8-sig')
                mapped_df = full_df[list(found_cols.keys())].rename(columns=found_cols)
                collected_dfs.append(mapped_df)

                for alias, target in found_cols.items():
                    if target not in source_report:
                        source_report[target] = []
                    source_report[target].append(f"{file} (gốc: {alias})")
        except Exception:
            continue

    if not collected_dfs:
        print("Không tìm thấy file nào chứa dữ liệu customer.")
        return

    final_df = pd.concat(collected_dfs, ignore_index=True)
    initial_len = len(final_df)
    final_df = final_df.drop_duplicates()
    print(f"\nĐã xử lý: Xóa {initial_len - len(final_df)} dòng trùng lặp.")

    for col in target_cols:
        if col not in final_df.columns:
            final_df[col] = None

    defaults = {
        'customer_id': 'UNKNOWN_ID', 'zip_FK': '00000', 'signup_date': date.today(),
        'gender': 'Unknown', 'age_group': 'Unknown', 'acq_channel': 'Unknown'
    }
    for col, val in defaults.items():
        final_df[col] = final_df[col].fillna(val)

    final_df['signup_date'] = pd.to_datetime(final_df['signup_date'], errors='coerce').dt.date.fillna(date.today())

    # Lưu kết quả
    output_file = '/content/customers_new.csv'
    final_df[target_cols].to_csv(output_file, index=False, encoding='utf-8-sig')

    print(f"\n--- HOÀN THÀNH ---")
    print(f"File lưu tại: {output_file}")

    print("\n--- BÁO CÁO NGUỒN DỮ LIỆU (SOURCE REPORT) ---")
    for col in target_cols:
        sources = ", ".join(source_report.get(col, ["Không tìm thấy - Sử dụng mặc định"]))
        print(f"Cột '{col}': {sources}")

scan_and_process_customers()